# UNet with Multi-Level Conditioning (MLC)

## Goal

Modify this UNet (originally from OpenAI's guided-diffusion, via TorchCFM) to support **spatial FiLM conditioning** from multi-level PCA embeddings derived from a Swin V2 transformer encoder. The original UNet only supports class conditioning (added to the time embedding) and has no mechanism for injecting spatially-structured conditioning at intermediate layers.

## Conditioning Plan

**Encoder-side spatial FiLM** at matching resolutions, plus global conditioning on the time embedding:

| Conditioning | Target | Method | Notes |
|---|---|---|---|
| Stage 0 (32×32, 3 ch) | L2 enc (32×32, 256 ch) | Spatial FiLM | Exact spatial match |
| Stage 1 (16×16, 7 ch) | L3 enc (16×16, 256 ch) | Spatial FiLM | Exact spatial match |
| Stage 2 (8×8, 23 ch) | L4 enc (8×8, 256 ch) | Spatial FiLM | Exact spatial match |
| Stages 2–4 (→8×8 after upsample) | Bottleneck (8×8, 256 ch) | Spatial FiLM | Stage 3 (4×4) upsampled 2×, Stage 4 (2×2) upsampled 4× |
| Stage 5 (1×1, 17 ch) | Time embedding (512-d) | Linear → add | Global signal, same pattern as class conditioning |

L0 (128×128) and L1 (64×64) are finer than any Swin patch level, so they receive no conditioning.

## UNet Architecture Reference

With `num_channel=128`, `channel_mult=[1,2,2,2,2]`, input 128×128:

| Stage | Spatial | Channels |
|---|---|---|
| Input | 128×128 | 1 |
| L0 (enc) | 128×128 | 128 |
| L1 (enc) | 64×64 | 256 |
| L2 (enc) | 32×32 | 256 |
| L3 (enc) | 16×16 | 256 (attention here) |
| L4 (enc) | 8×8 | 256 |
| Bottleneck | 8×8 | 256 |

## PCA Conditioning Geometry

| Stage | Patches | Spatial grid | PCA comps | Shape |
|---|---|---|---|---|
| Stage 0 | 1024 | 32×32 | 3 | (32×32, 3) |
| Stage 1 | 256 | 16×16 | 7 | (16×16, 7) |
| Stage 2 | 64 | 8×8 | 23 | (8×8, 23) |
| Stage 3 | 16 | 4×4 | 37 | (4×4, 37) |
| Stage 4 | 4 | 2×2 | 31 | (2×2, 31) |
| Stage 5 | 1 | 1×1 | 17 | (1×1, 17) |

## Implementation Approach

**Spatial FiLM** at each injection point: interpolate conditioning to match spatial resolution → `Conv2d(cond_channels, 2 * unet_channels)` → split into `(scale, shift)` → `h = h * (1 + scale) + shift`.

Key touchpoints to modify:
1. **`TimestepEmbedSequential`** — needs to accept and pass through a `cond` argument alongside `(x, emb)`
2. **`ResBlock`** (or a new wrapper) — apply FiLM after the existing time-embedding injection
3. **`UNetModel.forward`** — thread the conditioning list through the encoder loop, applying each level at the right resolution
4. **`UNetModelWrapper`** — accept conditioning specs, build FiLM projection layers, expose in `forward`

Decoder-side conditioning is intentionally omitted (VRAM constraints); skip connections carry conditioned encoder features to the decoder.

## Original Source

Copied from `torchcfm/models/unet/unet.py` (OpenAI guided-diffusion UNet). Classes not relevant to the main UNet path (`SuperResModel`, `EncoderUNetModel`) are included but will not be modified.


In [ ]:
#| default_exp unet_mlc

In [ ]:
#| export
"""From https://raw.githubusercontent.com/openai/guided-diffusion/main/guided_diffusion/unet.py."""

import math
from abc import abstractmethod

import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F

from .fp16_util import convert_module_to_f16, convert_module_to_f32
from .nn import (
    avg_pool_nd,
    checkpoint,
    conv_nd,
    linear,
    normalization,
    timestep_embedding,
    zero_module,
)

ImportError: attempted relative import with no known parent package

In [ ]:
#| export
class AttentionPool2d(nn.Module):
    """Adapted from CLIP: https://github.com/openai/CLIP/blob/main/clip/model.py."""

    def __init__(
        self,
        spacial_dim: int,
        embed_dim: int,
        num_heads_channels: int,
        output_dim: int = None,
    ):
        super().__init__()
        self.positional_embedding = nn.Parameter(
            th.randn(embed_dim, spacial_dim**2 + 1) / embed_dim**0.5
        )
        self.qkv_proj = conv_nd(1, embed_dim, 3 * embed_dim, 1)
        self.c_proj = conv_nd(1, embed_dim, output_dim or embed_dim, 1)
        self.num_heads = embed_dim // num_heads_channels
        self.attention = QKVAttention(self.num_heads)

    def forward(self, x):
        b, c, *_spatial = x.shape
        x = x.reshape(b, c, -1)  # NC(HW)
        x = th.cat([x.mean(dim=-1, keepdim=True), x], dim=-1)  # NC(HW+1)
        x = x + self.positional_embedding[None, :, :].to(x.dtype)  # NC(HW+1)
        x = self.qkv_proj(x)
        x = self.attention(x)
        x = self.c_proj(x)
        return x[:, :, 0]

In [ ]:
#| export
class TimestepBlock(nn.Module):
    """Any module where forward() takes timestep embeddings as a second argument."""

    @abstractmethod
    def forward(self, x, emb):
        """Apply the module to `x` given `emb` timestep embeddings."""

In [ ]:
#| export
class SpatialFiLM(nn.Module):
    """Spatially-adaptive FiLM conditioning: projects a conditioning map to scale & shift."""

    def __init__(self, cond_channels, out_channels):
        super().__init__()
        self.proj = zero_module(nn.Conv2d(cond_channels, 2 * out_channels, 1))

    def forward(self, x, cond):
        # Interpolate cond to match x's spatial dims if needed
        if cond.shape[2:] != x.shape[2:]:
            cond = F.interpolate(cond, size=x.shape[2:], mode="bilinear", align_corners=False)
        scale, shift = self.proj(cond).chunk(2, dim=1)
        return x * (1 + scale) + shift


In [ ]:
#| export
class TimestepEmbedSequential(nn.Sequential, TimestepBlock):
    """A sequential module that passes timestep embeddings to the children that support it as an
    extra input."""

    def forward(self, x, emb, cond=None):
        for layer in self:
            if isinstance(layer, SpatialFiLM):
                x = layer(x, cond)
            elif isinstance(layer, TimestepBlock):
                x = layer(x, emb)
            else:
                x = layer(x)
        return x

In [ ]:
#| export
class Upsample(nn.Module):
    """An upsampling layer with an optional convolution.

    :param channels: channels in the inputs and outputs.
    :param use_conv: a bool determining if a convolution is applied.
    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then upsampling occurs in the
        inner-two dimensions.
    """

    def __init__(self, channels, use_conv, dims=2, out_channels=None):
        super().__init__()
        self.channels = channels
        self.out_channels = out_channels or channels
        self.use_conv = use_conv
        self.dims = dims
        if use_conv:
            self.conv = conv_nd(dims, self.channels, self.out_channels, 3, padding=1)

    def forward(self, x):
        assert x.shape[1] == self.channels
        if self.dims == 3:
            x = F.interpolate(x, (x.shape[2], x.shape[3] * 2, x.shape[4] * 2), mode="nearest")
        else:
            x = F.interpolate(x, scale_factor=2, mode="nearest")
        if self.use_conv:
            x = self.conv(x)
        return x

In [ ]:
#| export
class Downsample(nn.Module):
    """A downsampling layer with an optional convolution.

    :param channels: channels in the inputs and outputs.
    :param use_conv: a bool determining if a convolution is applied.
    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then downsampling occurs in the
        inner-two dimensions.
    """

    def __init__(self, channels, use_conv, dims=2, out_channels=None):
        super().__init__()
        self.channels = channels
        self.out_channels = out_channels or channels
        self.use_conv = use_conv
        self.dims = dims
        stride = 2 if dims != 3 else (1, 2, 2)
        if use_conv:
            self.op = conv_nd(dims, self.channels, self.out_channels, 3, stride=stride, padding=1)
        else:
            assert self.channels == self.out_channels
            self.op = avg_pool_nd(dims, kernel_size=stride, stride=stride)

    def forward(self, x):
        assert x.shape[1] == self.channels
        return self.op(x)

In [ ]:
#| export
class ResBlock(TimestepBlock):
    """A residual block that can optionally change the number of channels.

    :param channels: the number of input channels.
    :param emb_channels: the number of timestep embedding channels.
    :param dropout: the rate of dropout.
    :param out_channels: if specified, the number of out channels.
    :param use_conv: if True and out_channels is specified, use a spatial convolution instead of a
        smaller 1x1 convolution to change the channels in the skip connection.
    :param dims: determines if the signal is 1D, 2D, or 3D.
    :param use_checkpoint: if True, use gradient checkpointing on this module.
    :param up: if True, use this block for upsampling.
    :param down: if True, use this block for downsampling.
    """

    def __init__(
        self,
        channels,
        emb_channels,
        dropout,
        out_channels=None,
        use_conv=False,
        use_scale_shift_norm=False,
        dims=2,
        use_checkpoint=False,
        up=False,
        down=False,
    ):
        super().__init__()
        self.channels = channels
        self.emb_channels = emb_channels
        self.dropout = dropout
        self.out_channels = out_channels or channels
        self.use_conv = use_conv
        self.use_checkpoint = use_checkpoint
        self.use_scale_shift_norm = use_scale_shift_norm

        self.in_layers = nn.Sequential(
            normalization(channels),
            nn.SiLU(),
            conv_nd(dims, channels, self.out_channels, 3, padding=1),
        )

        self.updown = up or down

        if up:
            self.h_upd = Upsample(channels, False, dims)
            self.x_upd = Upsample(channels, False, dims)
        elif down:
            self.h_upd = Downsample(channels, False, dims)
            self.x_upd = Downsample(channels, False, dims)
        else:
            self.h_upd = self.x_upd = nn.Identity()

        self.emb_layers = nn.Sequential(
            nn.SiLU(),
            linear(
                emb_channels,
                2 * self.out_channels if use_scale_shift_norm else self.out_channels,
            ),
        )
        self.out_layers = nn.Sequential(
            normalization(self.out_channels),
            nn.SiLU(),
            nn.Dropout(p=dropout),
            zero_module(conv_nd(dims, self.out_channels, self.out_channels, 3, padding=1)),
        )

        if self.out_channels == channels:
            self.skip_connection = nn.Identity()
        elif use_conv:
            self.skip_connection = conv_nd(dims, channels, self.out_channels, 3, padding=1)
        else:
            self.skip_connection = conv_nd(dims, channels, self.out_channels, 1)

    def forward(self, x, emb):
        """Apply the block to a Tensor, conditioned on a timestep embedding.

        :param x: an [N x C x ...] Tensor of features.
        :param emb: an [N x emb_channels] Tensor of timestep embeddings.
        :return: an [N x C x ...] Tensor of outputs.
        """
        return checkpoint(self._forward, (x, emb), self.parameters(), self.use_checkpoint)

    def _forward(self, x, emb):
        if self.updown:
            in_rest, in_conv = self.in_layers[:-1], self.in_layers[-1]
            h = in_rest(x)
            h = self.h_upd(h)
            x = self.x_upd(x)
            h = in_conv(h)
        else:
            h = self.in_layers(x)
        emb_out = self.emb_layers(emb).type(h.dtype)
        while len(emb_out.shape) < len(h.shape):
            emb_out = emb_out[..., None]
        if self.use_scale_shift_norm:
            out_norm, out_rest = self.out_layers[0], self.out_layers[1:]
            scale, shift = th.chunk(emb_out, 2, dim=1)
            h = out_norm(h) * (1 + scale) + shift
            h = out_rest(h)
        else:
            h = h + emb_out
            h = self.out_layers(h)
        return self.skip_connection(x) + h

In [ ]:
#| export
class AttentionBlock(nn.Module):
    """An attention block that allows spatial positions to attend to each other.

    Originally ported from here, but adapted to the N-d case.
    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L66.
    """

    def __init__(
        self,
        channels,
        num_heads=1,
        num_head_channels=-1,
        use_checkpoint=False,
        use_new_attention_order=False,
    ):
        super().__init__()
        self.channels = channels
        if num_head_channels == -1:
            self.num_heads = num_heads
        else:
            assert (
                channels % num_head_channels == 0
            ), f"q,k,v channels {channels} is not divisible by num_head_channels {num_head_channels}"
            self.num_heads = channels // num_head_channels
        self.use_checkpoint = use_checkpoint
        self.norm = normalization(channels)
        self.qkv = conv_nd(1, channels, channels * 3, 1)
        if use_new_attention_order:
            # split qkv before split heads
            self.attention = QKVAttention(self.num_heads)
        else:
            # split heads before split qkv
            self.attention = QKVAttentionLegacy(self.num_heads)

        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))

    def forward(self, x):
        return checkpoint(self._forward, (x,), self.parameters(), self.use_checkpoint)

    def _forward(self, x):
        b, c, *spatial = x.shape
        x = x.reshape(b, c, -1)
        qkv = self.qkv(self.norm(x))
        h = self.attention(qkv)
        h = self.proj_out(h)
        return (x + h).reshape(b, c, *spatial)

In [ ]:
#| export
def count_flops_attn(model, _x, y):
    """A counter for the `thop` package to count the operations in an attention operation.

    Meant to be used like:
        macs, params = thop.profile(
            model,
            inputs=(inputs, timestamps),
            custom_ops={QKVAttention: QKVAttention.count_flops},
        )
    """
    b, c, *spatial = y[0].shape
    num_spatial = int(np.prod(spatial))
    # We perform two matmuls with the same number of ops.
    # The first computes the weight matrix, the second computes
    # the combination of the value vectors.
    matmul_ops = 2 * b * (num_spatial**2) * c
    model.total_ops += th.DoubleTensor([matmul_ops])

In [ ]:
#| export
class QKVAttentionLegacy(nn.Module):
    """A module which performs QKV attention.

    Matches legacy QKVAttention + input/output heads shaping
    """

    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads

    def forward(self, qkv):
        """Apply QKV attention.

        :param qkv: an [N x (H * 3 * C) x T] tensor of Qs, Ks, and Vs.
        :return: an [N x (H * C) x T] tensor after attention.
        """
        bs, width, length = qkv.shape
        assert width % (3 * self.n_heads) == 0
        ch = width // (3 * self.n_heads)
        q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)
        scale = 1 / math.sqrt(math.sqrt(ch))
        weight = th.einsum(
            "bct,bcs->bts", q * scale, k * scale
        )  # More stable with f16 than dividing afterwards
        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)
        a = th.einsum("bts,bcs->bct", weight, v)
        return a.reshape(bs, -1, length)

    @staticmethod
    def count_flops(model, _x, y):
        return count_flops_attn(model, _x, y)

In [ ]:
#| export
class QKVAttention(nn.Module):
    """A module which performs QKV attention and splits in a different order."""

    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads

    def forward(self, qkv):
        """Apply QKV attention.

        :param qkv: an [N x (3 * H * C) x T] tensor of Qs, Ks, and Vs.
        :return: an [N x (H * C) x T] tensor after attention.
        """
        bs, width, length = qkv.shape
        assert width % (3 * self.n_heads) == 0
        ch = width // (3 * self.n_heads)
        q, k, v = qkv.chunk(3, dim=1)
        scale = 1 / math.sqrt(math.sqrt(ch))
        weight = th.einsum(
            "bct,bcs->bts",
            (q * scale).view(bs * self.n_heads, ch, length),
            (k * scale).view(bs * self.n_heads, ch, length),
        )  # More stable with f16 than dividing afterwards
        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)
        a = th.einsum("bts,bcs->bct", weight, v.reshape(bs * self.n_heads, ch, length))
        return a.reshape(bs, -1, length)

    @staticmethod
    def count_flops(model, _x, y):
        return count_flops_attn(model, _x, y)

In [ ]:
#| export
class UNetModelMLC(nn.Module):
    """The full UNet model with attention and timestep embedding.

    :param in_channels: channels in the input Tensor.
    :param model_channels: base channel count for the model.
    :param out_channels: channels in the output Tensor.
    :param num_res_blocks: number of residual blocks per downsample.
    :param attention_resolutions: a collection of downsample rates at which attention will take
        place. May be a set, list, or tuple. For example, if this contains 4, then at 4x
        downsampling, attention will be used.
    :param dropout: the dropout probability.
    :param channel_mult: channel multiplier for each level of the UNet.
    :param conv_resample: if True, use learned convolutions for upsampling and downsampling.
    :param dims: determines if the signal is 1D, 2D, or 3D.
    :param num_classes: if specified (as an int), then this model will be class-conditional with
        `num_classes` classes.
    :param use_checkpoint: use gradient checkpointing to reduce memory usage.
    :param num_heads: the number of attention heads in each attention layer.
    :param num_heads_channels: if specified, ignore num_heads and instead use a fixed channel width
        per attention head.
    :param num_heads_upsample: works with num_heads to set a different number of heads for
        upsampling. Deprecated.
    :param use_scale_shift_norm: use a FiLM-like conditioning mechanism.
    :param resblock_updown: use residual blocks for up/downsampling.
    :param use_new_attention_order: use a different attention pattern for potentially increased
        efficiency.
    :param mlcond_shapes: list (or ordered dict) of tuples of shapes for multi-level conditioning
    """

    def __init__(
        self,
        image_size,
        in_channels,
        model_channels,
        out_channels,
        num_res_blocks,
        attention_resolutions,
        dropout=0,
        channel_mult=(1, 2, 4, 8),
        conv_resample=True,
        dims=2,
        num_classes=None,
        use_checkpoint=False,
        use_fp16=False,
        num_heads=1,
        num_head_channels=-1,
        num_heads_upsample=-1,
        use_scale_shift_norm=False,
        resblock_updown=False,
        use_new_attention_order=False,
        mlcond_shapes=None,
    ):
        super().__init__()

        if num_heads_upsample == -1:
            num_heads_upsample = num_heads

        self.image_size = image_size
        self.in_channels = in_channels
        self.model_channels = model_channels
        self.out_channels = out_channels
        self.num_res_blocks = num_res_blocks
        self.attention_resolutions = attention_resolutions
        self.dropout = dropout
        self.channel_mult = channel_mult
        self.conv_resample = conv_resample
        self.num_classes = num_classes
        self.use_checkpoint = use_checkpoint
        self.dtype = th.float16 if use_fp16 else th.float32
        self.num_heads = num_heads
        self.num_head_channels = num_head_channels
        self.num_heads_upsample = num_heads_upsample

        time_embed_dim = model_channels * 4
        self.time_embed = nn.Sequential(
            linear(model_channels, time_embed_dim),
            nn.SiLU(),
            linear(time_embed_dim, time_embed_dim),
        )

        if self.num_classes is not None:
            self.label_emb = nn.Embedding(num_classes, time_embed_dim)

        ch = input_ch = int(channel_mult[0] * model_channels)
        self.input_blocks = nn.ModuleList(
            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]
        )
        self._feature_size = ch
        input_block_chans = [ch]
        ds = 1
        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [
                    ResBlock(
                        ch,
                        time_embed_dim,
                        dropout,
                        out_channels=int(mult * model_channels),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(mult * model_channels)
                print(f"DEBUG: image_size // ds  = ",image_size // ds)
                if image_size // ds == 32: #insert a SpatialFiLM(3, ch) into the layers list.
                   cond_dim = 3 # mlcond_shapes['L5'][0] # TODO: don't hard code this
                   layers.append(SpatialFiLM(cond_dim, ch)) 

                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch
                input_block_chans.append(ch)
            if level != len(channel_mult) - 1:
                out_ch = ch
                self.input_blocks.append(
                    TimestepEmbedSequential(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels=out_ch,
                            dims=dims,
                            use_checkpoint=use_checkpoint,
                            use_scale_shift_norm=use_scale_shift_norm,
                            down=True,
                        )
                        if resblock_updown
                        else Downsample(ch, conv_resample, dims=dims, out_channels=out_ch)
                    )
                )
                ch = out_ch
                input_block_chans.append(ch)
                ds *= 2
                self._feature_size += ch

        self.middle_block = TimestepEmbedSequential(
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
            AttentionBlock(
                ch,
                use_checkpoint=use_checkpoint,
                num_heads=num_heads,
                num_head_channels=num_head_channels,
                use_new_attention_order=use_new_attention_order,
            ),
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
        )
        self._feature_size += ch

        self.output_blocks = nn.ModuleList([])
        for level, mult in list(enumerate(channel_mult))[::-1]:
            for i in range(num_res_blocks + 1):
                ich = input_block_chans.pop()
                layers = [
                    ResBlock(
                        ch + ich,
                        time_embed_dim,
                        dropout,
                        out_channels=int(model_channels * mult),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(model_channels * mult)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads_upsample,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                if level and i == num_res_blocks:
                    out_ch = ch
                    layers.append(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels=out_ch,
                            dims=dims,
                            use_checkpoint=use_checkpoint,
                            use_scale_shift_norm=use_scale_shift_norm,
                            up=True,
                        )
                        if resblock_updown
                        else Upsample(ch, conv_resample, dims=dims, out_channels=out_ch)
                    )
                    ds //= 2
                self.output_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch

        self.out = nn.Sequential(
            normalization(ch),
            nn.SiLU(),
            zero_module(conv_nd(dims, input_ch, out_channels, 3, padding=1)),
        )

    def convert_to_fp16(self):
        """Convert the torso of the model to float16."""
        self.input_blocks.apply(convert_module_to_f16)
        self.middle_block.apply(convert_module_to_f16)
        self.output_blocks.apply(convert_module_to_f16)

    def convert_to_fp32(self):
        """Convert the torso of the model to float32."""
        self.input_blocks.apply(convert_module_to_f32)
        self.middle_block.apply(convert_module_to_f32)
        self.output_blocks.apply(convert_module_to_f32)

    def forward(self, t, x, y=None, mlcond=None):
        """Apply the model to an input batch.

        :param x: an [N x C x ...] Tensor of inputs.
        :param timesteps: a 1-D batch of timesteps.
        :param y: an [N] Tensor of labels, if class-conditional.
        :param mlcond: a dict or list. Multi-level conditioning signal 
        :return: an [N x C x ...] Tensor of outputs.
        """
        timesteps = t
        assert (y is not None) == (
            self.num_classes is not None
        ), "must specify y if and only if the model is class-conditional"
        while timesteps.dim() > 1:
            print(timesteps.shape)
            timesteps = timesteps[:, 0]
        if timesteps.dim() == 0:
            timesteps = timesteps.repeat(x.shape[0])

        hs = []
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))

        if self.num_classes is not None:
            assert y.shape == (x.shape[0],)
            emb = emb + self.label_emb(y)

        h = x.type(self.dtype)
        for module in self.input_blocks:
            h = module(h, emb)
            hs.append(h)
        h = self.middle_block(h, emb)
        for module in self.output_blocks:
            h = th.cat([h, hs.pop()], dim=1)
            h = module(h, emb)
        h = h.type(x.dtype)
        return self.out(h)

NameError: name 'nn' is not defined

In [ ]:
#| export
class SuperResModel(UNetModel):
    """A UNetModel that performs super-resolution.

    Expects an extra kwarg `low_res` to condition on a low-resolution image.
    """

    def __init__(self, image_size, in_channels, *args, **kwargs):
        super().__init__(image_size, in_channels * 2, *args, **kwargs)

    def forward(self, x, timesteps, low_res=None, **kwargs):
        _, _, new_height, new_width = x.shape
        upsampled = F.interpolate(low_res, (new_height, new_width), mode="bilinear")
        x = th.cat([x, upsampled], dim=1)
        return super().forward(x, timesteps, **kwargs)

In [ ]:
#| export
class EncoderUNetModelMLC(nn.Module):
    """The half UNet model with attention and timestep embedding.

    For usage, see UNet.
    """

    def __init__(
        self,
        image_size,
        in_channels,
        model_channels,
        out_channels,
        num_res_blocks,
        attention_resolutions,
        dropout=0,
        channel_mult=(1, 2, 4, 8),
        conv_resample=True,
        dims=2,
        use_checkpoint=False,
        use_fp16=False,
        num_heads=1,
        num_head_channels=-1,
        num_heads_upsample=-1,
        use_scale_shift_norm=False,
        resblock_updown=False,
        use_new_attention_order=False,
        pool="adaptive",
    ):
        super().__init__()

        if num_heads_upsample == -1:
            num_heads_upsample = num_heads

        self.in_channels = in_channels
        self.model_channels = model_channels
        self.out_channels = out_channels
        self.num_res_blocks = num_res_blocks
        self.attention_resolutions = attention_resolutions
        self.dropout = dropout
        self.channel_mult = channel_mult
        self.conv_resample = conv_resample
        self.use_checkpoint = use_checkpoint
        self.dtype = th.float16 if use_fp16 else th.float32
        self.num_heads = num_heads
        self.num_head_channels = num_head_channels
        self.num_heads_upsample = num_heads_upsample

        time_embed_dim = model_channels * 4
        self.time_embed = nn.Sequential(
            linear(model_channels, time_embed_dim),
            nn.SiLU(),
            linear(time_embed_dim, time_embed_dim),
        )

        ch = int(channel_mult[0] * model_channels)
        self.input_blocks = nn.ModuleList(
            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]
        )
        self._feature_size = ch
        input_block_chans = [ch]
        ds = 1
        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [
                    ResBlock(
                        ch,
                        time_embed_dim,
                        dropout,
                        out_channels=int(mult * model_channels),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(mult * model_channels)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch
                input_block_chans.append(ch)
            if level != len(channel_mult) - 1:
                out_ch = ch
                self.input_blocks.append(
                    TimestepEmbedSequential(
                        ResBlock(
                            ch,
                            time_embed_dim,
                            dropout,
                            out_channels=out_ch,
                            dims=dims,
                            use_checkpoint=use_checkpoint,
                            use_scale_shift_norm=use_scale_shift_norm,
                            down=True,
                        )
                        if resblock_updown
                        else Downsample(ch, conv_resample, dims=dims, out_channels=out_ch)
                    )
                )
                ch = out_ch
                input_block_chans.append(ch)
                ds *= 2
                self._feature_size += ch

        self.middle_block = TimestepEmbedSequential(
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
            AttentionBlock(
                ch,
                use_checkpoint=use_checkpoint,
                num_heads=num_heads,
                num_head_channels=num_head_channels,
                use_new_attention_order=use_new_attention_order,
            ),
            ResBlock(
                ch,
                time_embed_dim,
                dropout,
                dims=dims,
                use_checkpoint=use_checkpoint,
                use_scale_shift_norm=use_scale_shift_norm,
            ),
        )
        self._feature_size += ch
        self.pool = pool
        if pool == "adaptive":
            self.out = nn.Sequential(
                normalization(ch),
                nn.SiLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
                zero_module(conv_nd(dims, ch, out_channels, 1)),
                nn.Flatten(),
            )
        elif pool == "attention":
            assert num_head_channels != -1
            self.out = nn.Sequential(
                normalization(ch),
                nn.SiLU(),
                AttentionPool2d((image_size // ds), ch, num_head_channels, out_channels),
            )
        elif pool == "spatial":
            self.out = nn.Sequential(
                nn.Linear(self._feature_size, 2048),
                nn.ReLU(),
                nn.Linear(2048, self.out_channels),
            )
        elif pool == "spatial_v2":
            self.out = nn.Sequential(
                nn.Linear(self._feature_size, 2048),
                normalization(2048),
                nn.SiLU(),
                nn.Linear(2048, self.out_channels),
            )
        else:
            raise NotImplementedError(f"Unexpected {pool} pooling")

    def convert_to_fp16(self):
        """Convert the torso of the model to float16."""
        self.input_blocks.apply(convert_module_to_f16)
        self.middle_block.apply(convert_module_to_f16)

    def convert_to_fp32(self):
        """Convert the torso of the model to float32."""
        self.input_blocks.apply(convert_module_to_f32)
        self.middle_block.apply(convert_module_to_f32)

    def forward(self, x, timesteps):
        """Apply the model to an input batch.

        :param x: an [N x C x ...] Tensor of inputs.
        :param timesteps: a 1-D batch of timesteps.
        :return: an [N x K] Tensor of outputs.
        """
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))

        results = []
        h = x.type(self.dtype)
        for module in self.input_blocks:
            h = module(h, emb)
            if self.pool.startswith("spatial"):
                results.append(h.type(x.dtype).mean(dim=(2, 3)))
        h = self.middle_block(h, emb)
        if self.pool.startswith("spatial"):
            results.append(h.type(x.dtype).mean(dim=(2, 3)))
            h = th.cat(results, axis=-1)
            return self.out(h)
        else:
            h = h.type(x.dtype)
            return self.out(h)

In [ ]:
#| export
NUM_CLASSES = 1000

In [ ]:
#| export
class UNetModelWrapperMLC(UNetModelMLC):
    def __init__(
        self,
        dim,
        num_channels,
        num_res_blocks,
        channel_mult=None,
        learn_sigma=False,
        class_cond=False,
        num_classes=NUM_CLASSES,
        use_checkpoint=False,
        attention_resolutions="16",
        num_heads=1,
        num_head_channels=-1,
        num_heads_upsample=-1,
        use_scale_shift_norm=False,
        dropout=0,
        resblock_updown=False,
        use_fp16=False,
        use_new_attention_order=False,
    ):
        """Dim (tuple): (C, H, W)"""
        image_size = dim[-1]
        if channel_mult is None:
            if image_size == 512:
                channel_mult = (0.5, 1, 1, 2, 2, 4, 4)
            elif image_size == 256:
                channel_mult = (1, 1, 2, 2, 4, 4)
            elif image_size == 128:
                channel_mult = (1, 1, 2, 3, 4)
            elif image_size == 64:
                channel_mult = (1, 2, 3, 4)
            elif image_size == 32:
                channel_mult = (1, 2, 2, 2)
            elif image_size == 28:
                channel_mult = (1, 2, 2)
            else:
                raise ValueError(f"unsupported image size: {image_size}")
        else:
            channel_mult = list(channel_mult)

        attention_ds = []
        for res in attention_resolutions.split(","):
            attention_ds.append(image_size // int(res))

        return super().__init__(
            image_size=image_size,
            in_channels=dim[0],
            model_channels=num_channels,
            out_channels=(dim[0] if not learn_sigma else dim[0] * 2),
            num_res_blocks=num_res_blocks,
            attention_resolutions=tuple(attention_ds),
            dropout=dropout,
            channel_mult=channel_mult,
            num_classes=(num_classes if class_cond else None),
            use_checkpoint=use_checkpoint,
            use_fp16=use_fp16,
            num_heads=num_heads,
            num_head_channels=num_head_channels,
            num_heads_upsample=num_heads_upsample,
            use_scale_shift_norm=use_scale_shift_norm,
            resblock_updown=resblock_updown,
            use_new_attention_order=use_new_attention_order,
        )

    def forward(self, t, x, y=None, *args, **kwargs):
        return super().forward(t, x, y=y)

Okay, please read this entire file, or maybe you're already reading it. I want you to help walk me through the modifications that we need to do.  Ultimately this is going to be used in place of the existing unit instantiation in `train_cfm_midi.py` Which is gonna supply the PCA embeddings from the different stages, although they're labeled as levels which is going to be a little confusing.  But my understanding is we're going to have to propagate the conditioning signal in from the forward call of the unit into all the various places where it needs to get injected. We're going to ignore the two finest levels of the unit encoder, and we're going to start matching up inside the third (L2) Part of the units encoder. So, what do we need to change first?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I've read through all the code and the conditioning plan. You've got a clear picture of what's needed. Let me walk you through it in order.

The **first thing to change** is `TimestepEmbedSequential`. Right now it only passes `(x, emb)` to its children. Every encoder block flows through this class, so it's the plumbing layer — if it can't carry a conditioning signal, nothing downstream can receive one.

The change is small: its `forward` needs to accept an optional `cond` argument and pass it through to any child layer that knows what to do with it (i.e., a FiLM layer we'll create next). Children that don't support it (like `ResBlock`, `AttentionBlock`, plain convs) keep working as before.

So the pattern would be:

```python
def forward(self, x, emb, cond=None):
    for layer in self:
        if isinstance(layer, SomeNewFiLMLayer):
            x = layer(x, cond)
        elif isinstance(layer, TimestepBlock):
            x = layer(x, emb)
        else:
            x = layer(x)
    return x
```

Before you code that though — do you want to define the FiLM conditioning module first (so we know what class to check for in that `isinstance`), or do you prefer to stub out `TimestepEmbedSequential` with a placeholder name and fill it in after?

Agreed. And it'd be nice if this film layer class could be sufficiently general that we could use it all over the place.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good call. The key design decisions for a general `SpatialFiLM` module:

1. **Constructor** takes `cond_channels` (varies per stage: 3, 7, 23, etc.) and `out_channels` (the UNet channel width at the injection point)
2. **1×1 Conv** projects `cond_channels → 2 * out_channels`, then chunk into `(scale, shift)`
3. **Auto-interpolation** — if the conditioning spatial dims don't match `x`, bilinear-interpolate to fit (this handles the bottleneck case where stages 3 and 4 need upsampling)
4. Apply: `x * (1 + scale) + shift`

The `(1 + scale)` initialization trick means that if the conv is zero-initialized, FiLM starts as an identity — the model trains normally before the conditioning signal has learned anything useful.

Two questions before you write it:

- Should the projection conv be zero-initialized (via `zero_module`) so it starts as identity, or do you want standard init?
- Does it need to be a `TimestepBlock` subclass, or should it be its own thing that `TimestepEmbedSequential` checks for separately (I'd lean toward the latter since it doesn't use `emb`)?

I'm not sure. I've written film layers before and they're usually really simple... Actually, the difference between this one and the others is going to be whether it's going to be spatial or just a regular linear projection. I don't think we're even doing a film layer for the time embedding, are we?  right? "Linear -> add" So if you're talking about the time, we're just going to do a linear kind of thing and add it. No film. Not yet, all the film stuff is spatial

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right, good clarification. So two distinct mechanisms:

1. **`SpatialFiLM`** — Conv2d-based, for stages 0–2 and the bottleneck. Takes spatial conditioning, projects to scale+shift, applies `x * (1 + scale) + shift`.

2. **Stage 5 (global)** — just a `nn.Linear(17, time_embed_dim)` whose output gets added to `emb`, same pattern as the existing `label_emb` class conditioning. No new class needed for that.

So `SpatialFiLM` can be really simple — just a `nn.Module` with a `Conv2d(cond_channels, 2 * out_channels, 1)` and the forward does interpolate-if-needed, chunk, and apply. Want to go ahead and write it?

Wait a second. if I screwed up. `TimestepBlock` doesn't seem to do anything in its forward.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right — `TimestepBlock` is just an abstract base class. It's a **marker interface**. It doesn't do anything itself; its only purpose is so that `TimestepEmbedSequential.forward` can check `isinstance(layer, TimestepBlock)` to decide whether to pass `emb` or not.

`ResBlock` inherits from it, which is why it receives `emb`. Plain layers like `Downsample` and `AttentionBlock` don't inherit from it, so they just get `x`.

So when we add `SpatialFiLM` later, we'll have a similar choice — either give it its own marker class, or just check `isinstance(layer, SpatialFiLM)` directly. Does that clear it up?

```python
class SpatialFilm(nn.Module): 
    def __init___(self): 
        self.conv = nn.Conv2d(stuff?)
        self.gamma = nn.Parameter()
        self.beta = nn.Parameter()
    def forward(self, x): 
        return self.gamma * self.conv(x) + self.beta 
```

...ish? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Close! A couple of things:

- You don't need separate `gamma`/`beta` parameters — the `Conv2d` itself outputs both scale and shift by projecting to `2 * out_channels`, then you `chunk` the result into two halves
- `forward` needs **two** inputs: `x` (the feature map being modulated) and `cond` (the conditioning signal being projected)
- The formula is `x * (1 + scale) + shift` — applied to `x`, not to the conv output

So the structure is more like:

```python
def forward(self, x, cond):
    # project cond, interpolate if needed, chunk into scale & shift, apply to x
```

Want to take another pass at it?

No, I'm tired of my brain hurts. And film layers are standard enough that if I could dig one out, I would copy and paste it in right here, right now. So, can you just write it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
class SpatialFiLM(nn.Module):
    """Spatially-adaptive FiLM conditioning: projects a conditioning map to scale & shift."""

    def __init__(self, cond_channels, out_channels):
        super().__init__()
        self.proj = zero_module(nn.Conv2d(cond_channels, 2 * out_channels, 1))

    def forward(self, x, cond):
        # Interpolate cond to match x's spatial dims if needed
        if cond.shape[2:] != x.shape[2:]:
            cond = F.interpolate(cond, size=x.shape[2:], mode="bilinear", align_corners=False)
        scale, shift = self.proj(cond).chunk(2, dim=1)
        return x * (1 + scale) + shift
```

`zero_module` initializes the conv to all zeros, so at the start of training `scale=0, shift=0` and FiLM is a no-op — the model behaves exactly like the unconditioned version until it learns to use the signal.

Okay, I'm going to copy that and paste it up near the top so we can use it throughout the file.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sounds good! Once you've got it placed, we can move on to the next step — which is updating `TimestepEmbedSequential.forward` to accept and pass through the conditioning to any `SpatialFiLM` layers it contains.

Okay, I went ahead and updated `TimestepEmbedSequential`'s forward method. What do you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looks great! The ordering of the `isinstance` checks is correct — `SpatialFiLM` first (since it's not a `TimestepBlock`), then `TimestepBlock` for things like `ResBlock`, then the fallback for plain layers like convs and attention.

Now every block in the UNet can carry `cond` through and hand it to a `SpatialFiLM` if one is present. Next step is wiring it up in `UNetModel` — both creating the FiLM layers in `__init__` and threading the conditioning through in `forward`. Want to tackle that?

Yeah, tell me what where to go next in the code and what we need to do there. For the sake of naming, we should probably change "UNet" to UNetMLC"? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea on the rename. I'd suggest:

- **`UNetModel`** — leave as-is (it's the base class, and we need to modify its `forward` to thread `cond` through, but it stays general)
- **`UNetModelWrapperMLC`** (or just `UNetMLC`) — replaces `UNetModelWrapper`, and this is where we build the FiLM layers and the stage 5 linear projection

The next place to touch is **`UNetModel.forward`**. Right now the encoder loop is:

```python
for module in self.input_blocks:
    h = module(h, emb)
```

We need it to pass `cond` as well, so each `TimestepEmbedSequential` block can hand it to any `SpatialFiLM` inside it. The question is — should `UNetModel.forward` accept a single `cond` dict/list and pick the right one per level, or should that logic live in the wrapper?

I'd lean toward having `UNetModel.forward` accept a simple `cond_list` — one entry per `input_block` (with `None` for blocks that get no conditioning) — and just pass it straight through. Then the wrapper is responsible for building that list from the PCA stages. What do you think?

Okay, here's the modification I made to UNET models forward: 
```python
    def forward(self, t, x, y=None, mlcond=None):
        """Apply the model to an input batch.

        :param x: an [N x C x ...] Tensor of inputs.
        :param timesteps: a 1-D batch of timesteps.
        :param y: an [N] Tensor of labels, if class-conditional.
        :param mlcond: a dict or list. Multi-level conditioning signal 
        :return: an [N x C x ...] Tensor of outputs.
        """
```
I want to be careful that we don't mix up our multi-level conditioning with the class conditioning that's already set up in there, hence "mlcond" instead of just "cond" Individual layers can have individual "cond", Passing into the unit, I wanted to make it clear from the get-go that this was going to be multi-level.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Smart naming choice — `mlcond` at the top level, `cond` at the individual layer level. That makes the distinction clear and avoids confusion with the class-conditional `y`.

Now the key question: inside that `forward`, we need to map from `mlcond` (the multi-level dict/list) to the right `cond` for each `input_block`. Looking at the encoder structure:

| Block index | What it is | Resolution | Conditioning? |
|---|---|---|---|
| 0 | input conv | 128×128 | None (L0) |
| 1–2 | L0 ResBlocks | 128×128 | None |
| 3 | Downsample | → 64×64 | None |
| 4–5 | L1 ResBlocks | 64×64 | None |
| 6 | Downsample | → 32×32 | None |
| 7–8 | L2 ResBlocks | 32×32 | **Stage 0** |
| 9 | Downsample | → 16×16 | None |
| 10–11 | L3 ResBlocks + Attn | 16×16 | **Stage 1** |
| 12 | Downsample | → 8×8 | None |
| 13–14 | L4 ResBlocks | 8×8 | **Stage 2** |
| middle | Bottleneck | 8×8 | **Stages 2–4 combined** |

So the first decision: should the mapping from block index → conditioning entry be hard-coded in `UNetModel.forward`, or should the wrapper pre-build a list with one entry per block? I'd lean toward the wrapper building the per-block list so `UNetModel.forward` stays simple — just `h = module(h, emb, cond=mlcond[i])`. What's your preference?

I confess that I'm feeling tired and your questions are passing over my head. What's the difference between the wrapper class and the unit model class? And honestly, why did you say we shouldn't change the class name? I think it should be name. UNetMLCModel or UnetModelMLC 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good questions. The difference is simple:

- **`UNetModel`** — the big class. Builds all the layers (encoder, bottleneck, decoder) and has the `forward` loop that runs data through them.
- **`UNetModelWrapper`** — a thin subclass that just picks sensible defaults (like `channel_mult` based on image size) and calls `UNetModel.__init__`. Its `forward` just calls `super().forward()`.

And you're right on the rename — since we're modifying `UNetModel.forward` to thread `mlcond` through, it *is* changing. Go ahead and rename it to `UNetModelMLC` (or whichever you prefer). Then the wrapper becomes `UNetModelWrapperMLC` or similar.

Okay, I appended "MLC" to the various relevant class names, just so that we don't accidentally mix up my new classes with the original ones.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good housekeeping. Now the next concrete step is in `UNetModelMLC.__init__` — we need to actually create the `SpatialFiLM` layers and append them into the right `TimestepEmbedSequential` blocks.

The tricky part: the FiLM layers need to be *inside* the `TimestepEmbedSequential` blocks at the right encoder levels, so that the existing `forward` loop picks them up automatically. That means we append a `SpatialFiLM` to the layers list when building blocks at L2, L3, and L4.

But to do that, `__init__` needs to know the conditioning spec — which stages go where and how many channels each has. Should that be passed in as a parameter (e.g. a list of `(cond_channels, target_resolution)` tuples), or hard-coded?

Great question. Okay, we're not trying to make a general library for everybody to use for anything. We're trying to get a job done for a particular task. For me.  That said, it would be nice if it were general and elegant and usable by others.  Would be great if it could just compare the dimensions of the conditioning and the various unit layers and just match them up wherever it can. So we could do a thing where it matches as much as it can. Anything left over that has more than one spatial dimension gets upsampled and connected to the bottleneck, and then the other, if there's a scalar or you know, multi-dimensional vector with no spatial extent, then that gets in with the time embedding.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a really nice design. So the logic in `__init__` would be:

1. **Build the encoder as usual**, tracking the resolution and channel count at each level
2. **Take a conditioning spec** — something like a list of `(H, W, C)` shapes
3. **For each conditioning entry**, compare its spatial dims against the encoder levels:
   - **Exact match** → create a `SpatialFiLM` and append it into that level's `TimestepEmbedSequential`
   - **Spatial but no match** → collect these for the bottleneck (upsample + combine → single `SpatialFiLM`)
   - **1×1** → linear projection, add to time embedding

The conditioning spec could be as simple as a list of tuples: `[(32,32,3), (16,16,7), (8,8,23), (4,4,37), (2,2,31), (1,1,17)]`.

Does that feel right as the interface, or would you rather pass it differently?

Yeah, that would be fine. We could even have it where we read a sample data point from the conditioning data to get the dimensions.   I think doing this would kind of bridge the gap between something that works only for me and something that would be general enough to work with other people. In that sense, maybe it's elegant. ;-) 


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I love it. So instead of a hand-written spec, you'd pass something like `cond_sample` — a list of tensors (or just their shapes) — and `__init__` figures out the wiring automatically. Very elegant, and zero chance of a mismatch between the spec and the actual data.

So the interface to `__init__` would just be one new parameter, something like:

```python
cond_shapes=None  # e.g. [(3,32,32), (7,16,16), (23,8,8), (37,4,4), (31,2,2), (17,1,1)]
```

(Channel-first to match PyTorch convention, batch dim omitted.)

Then the auto-matching logic runs after the encoder is built, since by that point we know the resolution at every level. Want to start writing that matching logic, or should I sketch it out?

By all means, sketch it out and then I'll probably go to bed and take a look at it again tomorrow when my brain is fresh.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the sketch. It goes in `UNetModelMLC.__init__`, after the encoder and bottleneck are already built:

```python
# --- Auto-match conditioning to encoder levels ---
# cond_shapes: list of (C, H, W) tuples, one per conditioning stage

if cond_shapes is not None:
    # 1. Build a map of resolution -> (block_indices, unet_channels) from encoder
    #    We tracked `ds` during construction; do the same thing again to build the map
    level_info = {}  # {spatial_size: (last_block_idx, ch)}
    ds = 1
    idx = 0
    for level, mult in enumerate(channel_mult):
        res = image_size // ds
        ch_at_level = int(mult * model_channels)
        for _ in range(num_res_blocks):
            idx += 1
            level_info.setdefault(res, (idx, ch_at_level))
        if level != len(channel_mult) - 1:
            idx += 1  # downsample block
            ds *= 2

    bottleneck_res = image_size // ds
    bottleneck_ch = int(channel_mult[-1] * model_channels)

    # 2. Sort each cond_shape into one of three buckets
    spatial_matched = []   # (cond_idx, block_idx, cond_ch, unet_ch)
    bottleneck_pool = []   # (cond_idx, cond_C, cond_H, cond_W)
    global_pool = []       # (cond_idx, cond_C)

    for i, (c, h, w) in enumerate(cond_shapes):
        if h == 1 and w == 1:
            global_pool.append((i, c))
        elif h in level_info:
            block_idx, unet_ch = level_info[h]
            spatial_matched.append((i, block_idx, c, unet_ch))
        else:
            bottleneck_pool.append((i, c, h, w))

    # 3. Spatial matches: append SpatialFiLM into the target input_block
    self.film_cond_map = {}  # block_idx -> cond_stage_index
    for cond_idx, block_idx, cond_ch, unet_ch in spatial_matched:
        film = SpatialFiLM(cond_ch, unet_ch)
        self.input_blocks[block_idx].append(film)
        self.film_cond_map[block_idx] = cond_idx

    # 4. Bottleneck pool: concat after upsampling, single SpatialFiLM
    if bottleneck_pool:
        total_cond_ch = sum(c for _, c, _, _ in bottleneck_pool)
        self.bottleneck_film = SpatialFiLM(total_cond_ch, bottleneck_ch)
        self.bottleneck_cond_indices = [i for i, _, _, _ in bottleneck_pool]
    else:
        self.bottleneck_film = None

    # 5. Global (1x1): linear projection added to time embedding
    if global_pool:
        total_global_ch = sum(c for _, c in global_pool)
        self.global_cond_proj = nn.Sequential(
            nn.Linear(total_global_ch, time_embed_dim), nn.SiLU()
        )
        self.global_cond_indices = [i for i, _ in global_pool]
    else:
        self.global_cond_proj = None
```

And the corresponding changes in `forward`:

```python
def forward(self, t, x, y=None, mlcond=None):
    # ... existing timestep/class embedding code ...

    # Global conditioning: add to emb
    if mlcond is not None and self.global_cond_proj is not None:
        gc = th.cat([mlcond[i].view(x.shape[0], -1) for i in self.global_cond_indices], dim=1)
        emb = emb + self.global_cond_proj(gc)

    # Encoder
    h = x.type(self.dtype)
    for i, module in enumerate(self.input_blocks):
        cond = None
        if mlcond is not None and i in self.film_cond_map:
            cond = mlcond[self.film_cond_map[i]]
        h = module(h, emb, cond=cond)
        hs.append(h)

    # Bottleneck
    if mlcond is not None and self.bottleneck_film is not None:
        bc = th.cat([
            F.interpolate(mlcond[i], size=h.shape[2:], mode="bilinear", align_corners=False)
            for i in self.bottleneck_cond_indices
        ], dim=1)
        h = self.middle_block(h, emb)
        h = self.bottleneck_film(h, bc)
    else:
        h = self.middle_block(h, emb)

    # Decoder unchanged
    for module in self.output_blocks:
        h = th.cat([h, hs.pop()], dim=1)
        h = module(h, emb)
    # ...
```

The key idea: `__init__` introspects the encoder's resolution map, compares against `cond_shapes`, and auto-wires everything. `forward` just looks up the pre-computed mapping. Sleep well and review it fresh tomorrow! 🌙

Wow, I had no idea it would involve that much code. I don't think your loop number one needs to be there at all. Because in building the layers of the unit, that geometry information will have already been calculated and created. You can just read that right off from the self model architecture 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're absolutely right. By that point in `__init__`, `input_block_chans` already has the channel count for every block, and `ds` was tracked throughout construction. So we can just build the `level_info` map *during* the existing encoder construction loop — one extra line to record `{resolution: (block_idx, ch)}` as we go — instead of re-deriving it after the fact.

That eliminates the whole "step 1" block. Want to fold that into the existing loop yourself, or want me to show where it'd slot in?

Yeah no, please retry the sketch you just made, only simplify it as I've described and in fact simplify it anywhere else you can. You get a lot of code there, man.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Fair point! Here's the slimmed version. During the existing encoder loop, just add one line alongside each `input_block_chans.append(ch)`:

```python
block_resolutions.append(image_size // ds)
```

Then after all the existing construction code:

```python
# --- Auto-wire conditioning ---
if cond_shapes is not None:
    # res -> (last_block_idx, unet_ch) from data we already collected
    res_map = {}
    for idx, (res, bch) in enumerate(zip(block_resolutions, input_block_chans)):
        res_map[res] = (idx, bch)

    self.film_cond_map = {}
    bottleneck_conds, global_conds = [], []

    for ci, (c, h, w) in enumerate(cond_shapes):
        if h == 1 and w == 1:
            global_conds.append((ci, c))
        elif h in res_map:
            block_idx, unet_ch = res_map[h]
            self.input_blocks[block_idx].append(SpatialFiLM(c, unet_ch))
            self.film_cond_map[block_idx] = ci
        else:
            bottleneck_conds.append((ci, c))

    self.bottleneck_film = (
        SpatialFiLM(sum(c for _, c in bottleneck_conds), ch)
        if bottleneck_conds else None)
    self.bottleneck_cond_indices = [i for i, _ in bottleneck_conds]

    self.global_cond_proj = (
        nn.Linear(sum(c for _, c in global_conds), time_embed_dim)
        if global_conds else None)
    self.global_cond_indices = [i for i, _ in global_conds]
```

And the `forward` changes:

```python
# After emb is computed:
if mlcond is not None and self.global_cond_proj is not None:
    gc = th.cat([mlcond[i].view(x.shape[0], -1) for i in self.global_cond_indices], dim=1)
    emb = emb + self.global_cond_proj(gc)

# Encoder — just add cond lookup
h = x.type(self.dtype)
for i, module in enumerate(self.input_blocks):
    cond = mlcond[self.film_cond_map[i]] if mlcond and i in self.film_cond_map else None
    h = module(h, emb, cond=cond)
    hs.append(h)

# Bottleneck
h = self.middle_block(h, emb)
if mlcond is not None and self.bottleneck_film is not None:
    bc = th.cat([F.interpolate(mlcond[i], size=h.shape[2:], mode="bilinear", align_corners=False)
                 for i in self.bottleneck_cond_indices], dim=1)
    h = self.bottleneck_film(h, bc)

# Decoder — unchanged
```

Much leaner — no redundant loops, and the matching logic reads naturally. Sleep on it and poke at it tomorrow! 🌙

yessssssss. That can be read by a human. Nice.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Thanks! 😄 It's way better when the code reads like a description of what it's doing rather than a puzzle to decode.

Get some rest — when you're back, the main tasks will be:

1. Wire `block_resolutions` into the existing encoder loop
2. Drop that auto-wiring block into `__init__`
3. Update `forward` with those few lines
4. Update the wrapper to pass `cond_shapes` through

Fresh eyes will make it quick work. 🌙

Good morning. I'm thinking let's start simple: let me do *only* the stage 0 link up and save the rest for later. baby steps, you know? so... how simple can we make this? ;-) 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good morning! Baby steps are the best steps.

For just Stage 0: it's a single `SpatialFiLM(3, 256)` appended into the L2 encoder block. No auto-matching logic, no bottleneck pooling, no global projection. You could literally hard-code it — one line in `__init__`, one extra arg in the `forward` loop. 

Want to figure out which block index is L2 and just jam it in there?

ooh yea i don't actually want to hard-code any dimensions! lol.  those values to be derived from elsewhere. 


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Smart. So during the encoder construction loop, you already know the resolution (`image_size // ds`). When you hit 32×32, that's your L2 block — grab its channel count and append a `SpatialFiLM`.

The channel count at that point is `int(channel_mult[level] * model_channels)`. For your config with `model_channels=128` and `channel_mult=[1,2,2,2,2]`, that'll be 256.

Want to add a simple check in the existing loop: "if resolution is 32×32, append FiLM and remember this block index"?

ok, i added kwarg: "mlcond_shapes=None" to the end of __init__.   now... where shall i declare the spatialfilm layers? before inserting them? e.g. after this part? 

```python
        ch = input_ch = int(channel_mult[0] * model_channels)
        self.input_blocks = nn.ModuleList(
            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]
        )
        self._feature_size = ch
        input_block_chans = [ch]
        ds = 1
```
maybe 
```python
      if mlcond_shapes:
        self.mlcond_layers = nn.ModuleList(
            [SpatialFiLM(x, self.input_blocks[i].shape) for x in mlcond_shapes for i in self.input_blocks if x==self._input_blocks[i].shape[1:] ]
        )
```
....? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Don't create them before the encoder's built — `self.input_blocks` doesn't exist yet at that point, and those layer objects don't have a `.shape` attribute anyway.

Two clean options for **Stage 0 only**:

**Option A:** During the existing encoder loop, when `image_size // ds == 32` (L2), append a `SpatialFiLM(3, ch)` right into that block's `layers` list before it gets wrapped in `TimestepEmbedSequential`.

**Option B:** After the entire encoder is built, iterate `self.input_blocks` to find which one has output resolution 32×32 (you could check by looking at what `ds` was when it was built, or just read the spatial size from a dummy forward), then `self.input_blocks[i].append(SpatialFiLM(3, 256))`.

Option A is probably easier since you already have `ds` and `ch` in hand. Which feels better?

> Don't create them before the encoder's built — self.input_blocks doesn't exist yet at that point, and those layer objects don't have a .shape attribute anyway.

?? isn't self.input_blocks created right before where I'm suggesting to add my code?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right that `self.input_blocks` exists there — but it only has **one element** (the initial input conv). The rest of the encoder blocks get added inside the `for level, mult in enumerate(channel_mult)` loop that comes after.

So at the point you suggested, you can't iterate over the full encoder because it hasn't been built yet.

For **Stage 0 only**, the cleanest baby step is to add the FiLM **inside that construction loop** when you hit 32×32 resolution. You already have `ds` (which tells you the current downsampling factor) and `ch` (the channel count) right there.

Want to add a simple check inside the existing loop instead?

oh you mean somewhere in this loop? 
```python
        ds = 1
        for level, mult in enumerate(channel_mult):
            for _ in range(num_res_blocks):
                layers = [
                    ResBlock(
                        ch,
                        time_embed_dim,
                        dropout,
                        out_channels=int(mult * model_channels),
                        dims=dims,
                        use_checkpoint=use_checkpoint,
                        use_scale_shift_norm=use_scale_shift_norm,
                    )
                ]
                ch = int(mult * model_channels)
                if ds in attention_resolutions:
                    layers.append(
                        AttentionBlock(
                            ch,
                            use_checkpoint=use_checkpoint,
                            num_heads=num_heads,
                            num_head_channels=num_head_channels,
                            use_new_attention_order=use_new_attention_order,
                        )
                    )
                self.input_blocks.append(TimestepEmbedSequential(*layers))
                self._feature_size += ch
                input_block_chans.append(ch)
```
...? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, exactly in that loop. Right after `layers = [ResBlock(...)]` and before the `TimestepEmbedSequential(*layers)` wrap, you can check if `image_size // ds == 32` and insert a `SpatialFiLM(3, ch)` into the `layers` list.

So the logic would be: "if resolution is 32×32 and `mlcond_shapes` is not None, append a FiLM layer to `layers` before it gets wrapped."

Want to try adding that check?

> Yes, exactly in that loop. Right after layers = [ResBlock(...)] and before the TimestepEmbedSequential(*layers) wrap, you can check if image_size // ds == 32 and insert a SpatialFiLM(3, ch) into the layers list.

what abut that AttentionBlock? do I do it before or after that? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

After the ResBlock, before AttentionBlock. That way the FiLM conditions the ResBlock output, then Attention works on the conditioned features.

So the order would be:
1. `ResBlock(...)`
2. `SpatialFiLM(3, ch)` ← add here
3. `AttentionBlock(...)` (if that resolution has attention)

Want to try inserting it there?

where are you getting the "3" from? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

From the Stage 0 conditioning spec — the table shows "Stage 0 (32×32, 3 ch)" meaning 3 PCA components at that resolution.

Want to derive it from `mlcond_shapes` instead so it's not hard-coded?

of course!  what do you think this means? 

> ooh yea i don't actually want to hard-code any dimensions! lol. those values to be derived from elsewhere.



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Oops, you literally just said that. My bad.

So we need to find the Stage 0 channel count from `mlcond_shapes` instead. The shapes are `(C, H, W)`, so you'd look for the entry where `H == 32 and W == 32` and grab its `C`.

Want to write a quick helper to find that, or should we assume `mlcond_shapes[0]` is Stage 0 for now and make it smarter later?

is there a way to check two elements of a list at once. e.g. if shape1[1:] == shape2{1:] then append spatialfilm(shape1[0], ch)   etc? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. In Python, tuple/list slices compare directly:

```python
if shape1[1:] == shape2[1:]:  # compares (H, W) == (H, W)
    # then use shape1[0] for channels
```

So in your case, you'd find which `mlcond_shape` has spatial dims matching `32, 32` (or whatever `image_size // ds` is), then use its first element for the channel count.

ok for now I"m going to semi-hard code it: 
```python
                print(f"DEBUG: image_size // ds  = ",image_size // ds)
                if image_size // ds == 32: #insert a SpatialFiLM(3, ch) into the layers list.
                   cond_dim = 3 # mlcond_shapes['L5'][0] # TODO: don't hard code this
                   layers.append(SpatialFiLM(cond_dim, ch))
``` 
lol

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect baby step! You've got the FiLM layer inserted at the right place in the encoder. The `layers.append(SpatialFiLM(cond_dim, ch))` goes right into that block's `TimestepEmbedSequential`.

Next up: updating `forward` to actually pass `mlcond` through and pick out the Stage 0 conditioning for that block. Or do you want to test that this at least instantiates without error first?